In [1]:
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd
import seaborn as sns
import os
import glob
from datetime import datetime
from datetime import timedelta
from matplotlib import pyplot as plt
import matplotlib.dates as md
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import warnings
from matplotlib import cm
import matplotlib.dates as mdates
from scipy.interpolate import interp2d
warnings.filterwarnings('ignore')
#import datetime
import scipy.ndimage as ndimage
from matplotlib import cm
import geopy.distance
#import matplotlib as mpl
from scipy.interpolate import interp1d
from sklearn.linear_model import LinearRegression
from shapely.geometry import Point
import geopandas as gpd
from geopandas import GeoDataFrame
import leafmap
import plotly.express as px
import matplotlib as mpl
import xarray as xr
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D
import math
#import pysplit
import netCDF4
import xarray as xr
import matplotlib.lines as mlines

c:\Users\taiwoajayi\Anaconda3\envs\TaiwoAjayi\lib\site-packages\pyproj\__init__.py:91: UserWarning: Valid PROJ data directory not found. Either set the path using the environmental variable PROJ_DATA (PROJ 9.1+) | PROJ_LIB (PROJ<9.1) or with `pyproj.datadir.set_data_dir`.
  warnings.warn(str(err))


In [2]:
csv_file_path = 'C:/Users/taiwoajayi/\OneDrive - University of Arizona/Arizona_ozone/Zip/Combined_Data.csv'

# Read the CSV file into a pandas DataFrame
MDA_O3 = pd.read_csv(csv_file_path, sep = ',', skiprows=0)
MDA_O3['Timestamp'] = pd.to_datetime(MDA_O3['Date Local'])
MD_O3 = MDA_O3.rename(columns={'1st Max Value': 'Sample Measurement', 'Arithmetic Mean': 'MDL'})
MD_O3['Sample Measurement'] = MD_O3['Sample Measurement'] * 1000

#MDA_O3['Sample Measurement'] = MDA_O3.apply(lambda row: row['Sample Measurement']*0.5*row['MDL'] if row['Sample Measurement'] < row['MDL'] else row['Sample Measurement'], axis=1)
MD_O3
# Define the columns of interest
columns_of_interest = ['Date Local', 'State Code', 'County Code', 'Site Num', 'Latitude', 'Longitude', 'Timestamp', 'Sample Measurement', 'MDL']

In [3]:
MD_O3

,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Sample Duration,...,Method Code,Method Name,Local Site Name,Address,State Name,County Name,City Name,CBSA Name,Date of Last Change,Timestamp
0,4,3,8001,44201,1,32.009410,-109.38906,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,47.0,INSTRUMENTAL - ULTRA VIOLET,Chiricahua NM - Entrance Station,CHIRICAHUA NATIONAL MOUMENT,Arizona,Cochise,Not in a city,"Sierra Vista-Douglas, AZ",2023-02-05,1997-01-01
1,4,3,8001,44201,1,32.009410,-109.38906,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,47.0,INSTRUMENTAL - ULTRA VIOLET,Chiricahua NM - Entrance Station,CHIRICAHUA NATIONAL MOUMENT,Arizona,Cochise,Not in a city,"Sierra Vista-Douglas, AZ",2023-02-05,1997-01-02
2,4,3,8001,44201,1,32.009410,-109.38906,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,47.0,INSTRUMENTAL - ULTRA VIOLET,Chiricahua NM - Entrance Station,CHIRICAHUA NATIONAL MOUMENT,Arizona,Cochise,Not in a city,"Sierra Vista-Douglas, AZ",2023-02-05,1997-01-03
3,4,3,8001,44201,1,32.009410,-109.38906,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,47.0,INSTRUMENTAL - ULTRA VIOLET,Chiricahua NM - Entrance Station,CHIRICAHUA NATIONAL MOUMENT,Arizona,Cochise,Not in a city,"Sierra Vista-Douglas, AZ",2023-02-05,1997-01-04
4,4,3,8001,44201,1,32.009410,-109.38906,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,47.0,INSTRUMENTAL - ULTRA VIOLET,Chiricahua NM - Entrance Station,CHIRICAHUA NATIONAL MOUMENT,Arizona,Cochise,Not in a city,"Sierra Vista-Douglas, AZ",2023-02-05,1997-01-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353296,4,27,8011,44201,1,32.690278,-114.61444,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,87.0,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,YUMA SUPERSITE,2323 S ARIZONA AVE,Arizona,Yuma,Yuma,"Yuma, AZ",2024-05-25,2022-12-27
353297,4,27,8011,44201,1,32.690278,-114.61444,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,87.0,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,YUMA SUPERSITE,2323 S ARIZONA AVE,Arizona,Yuma,Yuma,"Yuma, AZ",2024-05-25,2022-12-28
353298,4,27,8011,44201,1,32.690278,-114.61444,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,87.0,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,YUMA SUPERSITE,2323 S ARIZONA AVE,Arizona,Yuma,Yuma,"Yuma, AZ",2024-05-25,2022-12-29
353299,4,27,8011,44201,1,32.690278,-114.61444,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,87.0,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,YUMA SUPERSITE,2323 S ARIZONA AVE,Arizona,Yuma,Yuma,"Yuma, AZ",2024-05-25,2022-12-30


In [6]:
locations = {

    'Childrens Park': (-110.9823, -110.9823, 32.29515, 32.29515),
    'Green Valley': (-110.99644, -110.99644, 31.87952,  31.87952),
    'Coach Line': (-111.12716, -111.12716, 32.38082, 32.38082),
    'Rose Elementary': (-110.980134, -110.980134, 32.172995,  32.172995),
    'Fairgrounds': (-110.774357, -110.774357, 32.04767,  32.04767),
    'Tangerine': (-111.06352, -111.06352, 32.425261,  32.425261),
    'Craycroft': (-110.878067, -110.878067, 32.204411,  32.204411),
    'Saguaro Park': (-110.737116, -110.737116, 32.174538,  32.174538)
}

def compute_ozone_design_values(df, locations):
    df = df.copy()
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['Date'] = df['Timestamp'].dt.date
    df['Year'] = df['Timestamp'].dt.year

    # Assign location labels based on (lat, lon) match
    coord_to_name = {
        (round(lat, 5), round(lon, 5)): name
        for name, (lon, _, lat, _) in locations.items()
    }
    df['Location'] = df.apply(lambda row: coord_to_name.get(
        (round(row['Latitude'], 5), round(row['Longitude'], 5))
    ), axis=1)

    df = df.dropna(subset=['Location'])

    results = []

    for site in df['Location'].unique():
        site_df = df[df['Location'] == site]

        daily_max = site_df.groupby('Date')['Sample Measurement'].max().reset_index()
        daily_max['Year'] = pd.to_datetime(daily_max['Date']).dt.year

        # Get 4th highest per year
        fourth_highest = (
            daily_max.groupby('Year')['Sample Measurement']
            .apply(lambda x: np.sort(x)[-4] if len(x) >= 4 else np.nan)
        ).dropna()

        for year in range(fourth_highest.index.min() + 2, fourth_highest.index.max() + 1):
            years_used = [year - 2, year - 1, year]
            if all(y in fourth_highest.index for y in years_used):
                avg_dv = fourth_highest.loc[years_used].mean()
                results.append({
                    'Location': site,
                    'DV_Year': year,
                    'Design_Value_ppb': round(avg_dv, 1)
                })

    return pd.DataFrame(results)

dv_df = compute_ozone_design_values(MD_O3, locations)



In [7]:
dv_df

,Location,DV_Year,Design_Value_ppb
0,Saguaro Park,1999,75.0
1,Saguaro Park,2000,73.3
2,Saguaro Park,2001,70.0
3,Saguaro Park,2002,72.7
4,Saguaro Park,2003,74.0
...,...,...,...
169,Coach Line,2018,66.3
170,Coach Line,2019,67.3
171,Coach Line,2020,67.0
172,Coach Line,2021,66.3


In [12]:
# Filter by each range
dv_2001_2009 = dv_df[(dv_df['DV_Year'] >= 2001) & (dv_df['DV_Year'] <= 2009)]
dv_2010_2019 = dv_df[(dv_df['DV_Year'] >= 2010) & (dv_df['DV_Year'] <= 2019)]
dv_2020      = dv_df[dv_df['DV_Year'] == 2020]
dv_2021_2022 = dv_df[(dv_df['DV_Year'] >= 2021) & (dv_df['DV_Year'] <= 2022)]

# Group and calculate medians
med_2001_2009 = dv_2001_2009.groupby('Location')['Design_Value_ppb'].median().rename('DV_Median_01_09')
med_2010_2019 = dv_2010_2019.groupby('Location')['Design_Value_ppb'].median().rename('DV_Median_10_19')
med_2020      = dv_2020.groupby('Location')['Design_Value_ppb'].median().rename('DV_2020')
med_2021_2022 = dv_2021_2022.groupby('Location')['Design_Value_ppb'].median().rename('DV_Median_21_22')

# Combine into one table
dv_summary = pd.concat([med_2001_2009, med_2010_2019, med_2020, med_2021_2022], axis=1).reset_index()


In [13]:
dv_summary

,Location,DV_Median_01_09,DV_Median_10_19,DV_2020,DV_Median_21_22
0,Childrens Park,72.3,67.00,69.0,69.30
1,Coach Line,67.0,64.50,67.0,66.80
2,Craycroft,71.7,65.15,68.0,69.00
3,Fairgrounds,69.3,68.15,67.3,67.50
4,Green Valley,67.3,65.50,63.7,65.20
5,Rose Elementary,66.0,65.50,63.3,63.70
6,Saguaro Park,74.3,69.85,69.0,68.35
7,Tangerine,72.0,67.50,67.7,68.20


In [15]:
dv_summary.to_excel("C:/Users/taiwoajayi/\OneDrive - University of Arizona/Arizona_ozone/Zip/ozone_design_values_summary.xlsx", index=False)


In [18]:
# Pivot DV table: rows = Year, columns = Location
dv_summary = dv_df.pivot(index='DV_Year', columns='Location', values='Design_Value_ppb')
dv_summary = dv_summary.sort_index()  # Optional: sort years ascending
dv_summary.index.name = 'DV_Year'



dv_summary.to_excel("C:/Users/taiwoajayi/\OneDrive - University of Arizona/Arizona_ozone/Zip/ozone_design_values_summaryval.xlsx", index=True)

